In [1]:
# Q6 - Repeated contacts

# Question: where does extra calling stop paying off?

# `contacts_this_campaign` counts the contacts made to a customer in this campaign, including the last one. Customers who subscribe
# presumably stop being called, so the number of calls a customer received is partly a result of what they said, not only a cause of
# it. Each group is compared with the overall yes rate (the Q1 baseline) and shown with its customer count. Groups under 100
# customers are flagged and not interpreted.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q6, step 1: yes rate and calling effort by number of contacts in this campaign
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100
contacts = clean_df["contacts_this_campaign"]

edges = [1, 2, 3, 4, 5, 6, 11, 21, 31, 1000]
labels = ["1", "2", "3", "4", "5", "6-10", "11-20", "21-30", "31 and over"]
contact_band = pd.cut(contacts, bins=edges, labels=labels, right=False)

summary = (
    pd.DataFrame({"contact_band": contact_band, "subscribed": subscribed, "contacts": contacts})
    .groupby("contact_band", observed=True)
    .agg(customers=("subscribed", "count"), subscribers=("subscribed", "sum"), calls=("contacts", "sum"))
)
summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
summary["share_of_calls_pct"] = (summary["calls"] / contacts.sum() * 100).round(2)
summary["small_group"] = summary["customers"] < 100

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print(f"Customers in groups: {summary['customers'].sum()} (should be {len(clean_df)})")
print(f"Subscribers in groups: {summary['subscribers'].sum()} (should be {subscribed.sum()})")
print(f"Calls in groups: {summary['calls'].sum()} (should be {contacts.sum()})")
print(f"Customers with 11 or more contacts: {summary.loc[['11-20', '21-30', '31 and over'], 'customers'].sum()} (should be 869)")
print(f"Customers with 31 or more contacts: {summary.loc['31 and over', 'customers']} (should be 33)")
print()
summary

Baseline yes rate: 11.27%
Customers in groups: 41176 (should be 41176)
Subscribers in groups: 4639 (should be 4639)
Calls in groups: 105735 (should be 105735)
Customers with 11 or more contacts: 869 (should be 869)
Customers with 31 or more contacts: 33 (should be 33)



,customers,subscribers,calls,yes_rate_pct,vs_baseline_points,share_of_customers_pct,share_of_subscribers_pct,share_of_calls_pct,small_group
contact_band,,,,,,,,,
1,17634,2299,17634,13.04,1.77,42.83,49.56,16.68,False
2,10568,1211,21136,11.46,0.19,25.67,26.10,19.99,False
3,5340,574,16020,10.75,-0.52,12.97,12.37,15.15,False
4,2650,249,10600,9.40,-1.87,6.44,5.37,10.03,False
5,1599,120,7995,7.50,-3.77,3.88,2.59,7.56,False
6-10,2516,159,18274,6.32,-4.95,6.11,3.43,17.28,False
11-20,712,26,9864,3.65,-7.62,1.73,0.56,9.33,False
21-30,124,1,3035,0.81,-10.46,0.30,0.02,2.87,False
31 and over,33,0,1177,0.00,-11.27,0.08,0.00,1.11,True


In [4]:
# Q6, step 2: chance of a yes on each additional call
max_contacts = int(contacts.max())

rows = []
for call_number in range(1, max_contacts + 1):
    rows.append({
        "call_number": call_number,
        "calls_made": (contacts >= call_number).sum(),
        "subscriptions": ((contacts == call_number) & subscribed).sum(),
    })
per_call = pd.DataFrame(rows).set_index("call_number")

print(f"Calls made: {per_call['calls_made'].sum()} (should be {contacts.sum()})")
print(f"Subscriptions: {per_call['subscriptions'].sum()} (should be {subscribed.sum()})")
print(f"Calls made at number 1: {per_call.loc[1, 'calls_made']} (should be {len(clean_df)})")
print()

call_edges = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 16, 21, 31, 1000]
call_labels = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
               "11-15", "16-20", "21-30", "31 and over"]
per_call["call_group"] = pd.cut(per_call.index, bins=call_edges, labels=call_labels, right=False)

by_group = per_call.groupby("call_group", observed=True)[["calls_made", "subscriptions"]].sum()
by_group["yes_per_100_calls"] = (by_group["subscriptions"] / by_group["calls_made"] * 100).round(2)
by_group["cumulative_share_of_subscriptions_pct"] = (
    by_group["subscriptions"].cumsum() / subscribed.sum() * 100
).round(1)
by_group["cumulative_share_of_calls_pct"] = (
    by_group["calls_made"].cumsum() / contacts.sum() * 100
).round(1)
by_group["small_group"] = by_group["calls_made"] < 100

overall_per_call = subscribed.sum() / contacts.sum() * 100
print(f"Campaign-wide yes per 100 calls: {overall_per_call:.2f}")
print()
by_group

Calls made: 105735 (should be 105735)
Subscriptions: 4639 (should be 4639)
Calls made at number 1: 41176 (should be 41176)

Campaign-wide yes per 100 calls: 4.39



,calls_made,subscriptions,yes_per_100_calls,cumulative_share_of_subscriptions_pct,cumulative_share_of_calls_pct,small_group
call_group,,,,,,
1,41176,2299,5.58,49.6,38.9,False
2,23542,1211,5.14,75.7,61.2,False
3,12974,574,4.42,88.0,73.5,False
4,7634,249,3.26,93.4,80.7,False
5,4984,120,2.41,96.0,85.4,False
6,3385,75,2.22,97.6,88.6,False
7,2406,38,1.58,98.4,90.9,False
8,1777,17,0.96,98.8,92.6,False
9,1377,17,1.23,99.2,93.9,False
